In [6]:
import os, json, time
from dotenv import load_dotenv
load_dotenv()

PAGEINDEX_API_KEY = os.getenv("PAGEINDEX_API_KEY")
GROQ_API_KEY = os.getenv("GROQ_API_KEY")

print("PAGEINDEX_API_KEY Loaded:", "✅"  if PAGEINDEX_API_KEY else "PAGEINDEX_API_KEY Not Found")
print("GROQ_API_KEY Loaded:", "✅" if GROQ_API_KEY else "GROQ_API_KEY Not Found")

PAGEINDEX_API_KEY Loaded: ✅
GROQ_API_KEY Loaded: ✅


In [10]:
from groq import Groq
from pageindex import PageIndexClient

pi_client = PageIndexClient(api_key=PAGEINDEX_API_KEY)
groq_client = Groq(api_key=GROQ_API_KEY)

In [39]:
PDF_PATH = "Dilwale Dulhania Le Jayenge PDF.pdf"

result = pi_client.submit_document(file_path = PDF_PATH)
print("Document submitted.")
doc_id = result["doc_id"]
print("Document ID:", doc_id)

Document submitted.
Document ID: pi-cmpxtix2k03az01qunoptl94q


In [42]:
status_result = pi_client.get_document(doc_id)
status = status_result.get("status")
print(status)

completed


In [43]:
# Fetch the full tree 

tree_result = pi_client.get_tree(doc_id, node_summary = True)
pageindex_tree = tree_result.get("result", [])

print(f"Top level sections: {len(pageindex_tree)}")
print("Raw tree first node")
print(json.dumps(pageindex_tree[0] if pageindex_tree else {}, indent = 2))


Top level sections: 1
Raw tree first node
{
  "title": "Dilwale Dulhania Le Jayenge PDF",
  "node_id": "0000",
  "page_index": 1,
  "prefix_summary": "# Dilwale Dulhania Le Jayenge PDF\n\nAnupama Chopra\n\n![img-0.jpeg](img-0.jpeg)\n\nMore Free Books on Bookey\n\nScan to Download\n",
  "text": "# Dilwale Dulhania Le Jayenge PDF\n\nAnupama Chopra\n\n![img-0.jpeg](img-0.jpeg)\n\nMore Free Books on Bookey\n\nScan to Download\n",
  "nodes": [
    {
      "title": "Dilwale Dulhania Le Jayenge",
      "node_id": "0001",
      "page_index": 2,
      "summary": "## Dilwale Dulhania Le Jayenge\n\nThe Creation of a Bollywood Phenomenon\n\nWritten by Bookey\n\nCheck more about Dilwale Dulhania Le Jayenge Summary\n\nListen Dilwale Dulhania Le Jayenge Audiobook\n\nMore Free Books on Bookey\n\nScan to Download\n",
      "text": "## Dilwale Dulhania Le Jayenge\n\nThe Creation of a Bollywood Phenomenon\n\nWritten by Bookey\n\nCheck more about Dilwale Dulhania Le Jayenge Summary\n\nListen Dilwale Dulha

In [ ]:
# Pretty-print the full tree
def print_tree(nodes, indent=0):
    """Recursively print tree titles for a visual overview."""
    for node in nodes:
        prefix = "  " * indent + ("└─ " if indent > 0 else "")
        page   = node.get("page_index", "?")
        print(f"{prefix}[{node['node_id']}] {node['title']}  (p.{page})")
        if node.get("nodes"):
            print_tree(node["nodes"], indent + 1)

print("📚 Full Document Structure:\n")
print_tree(pageindex_tree)

📚 Full Document Structure:

[0000] Dilwale Dulhania Le Jayenge PDF  (p.1)
  └─ [0001] Dilwale Dulhania Le Jayenge  (p.2)
  └─ [0002] About the book  (p.3)
  └─ [0003] About the author  (p.4)
  └─ [0004] Summary Content List  (p.6)
  └─ [0005] Chapter 1: The Making of a Modern Classic - Origins and Inspirations  (p.7)
  └─ [0006] Chapter 2: Casting Choices and Iconic Performances - Building Unforgettable Characters  (p.11)
  └─ [0007] Chapter 3: Crafting the Perfect Love Story - Script, Dialogue, and Music  (p.14)
  └─ [0008] Chapter 4 : Filming Locations and Cinematography - A Visual Extravaganza  (p.17)
  └─ [0009] Chapter 5: Cultural Impact and Legacy - Dilwale Dulhania Le Jayenge’s Enduring Appeal  (p.20)
  └─ [0010] Chapter 6 : Behind the Scenes Stories - Anecdotes and Unseen Efforts  (p.24)
  └─ [0011] Chapter 7: The Phenomenon Lives On - Reflection on DDLJ's Timelessness  (p.27)


In [ ]:
# Count total nodes 
def count_nodes(nodes):
    total = len(nodes)
    for n in nodes:
        if n.get("nodes"):
            total += count_nodes(n["nodes"])
    return total

total = count_nodes(pageindex_tree)
print(f"🔢 Total nodes in tree: {total}")
print("   Each node = one retrievable section of the document")

🔢 Total nodes in tree: 12
   Each node = one retrievable section of the document


🧠 : LLM Tree Search — The Core of PageIndex
This is where PageIndex fundamentally differs from vector RAG.

Vector RAG retrieval:
- query → embed → cosine_similarity(query_vec, all_chunk_vecs) → top-k chunks
Problem: finds what's similar, not what's relevant

PageIndex retrieval:
- query + tree → LLM reasons → "node 0007 and 0008 contain the answer"
Advantage: LLM understands document structure, context, and intent

The LLM acts like a human expert scanning a Table of Contents.

In [52]:
# LLM tree search function

def llm_tree_search(query: str, tree: list, model: str = "llama-3.1-8b-instant") -> dict:
    """
    Core pageindex retrieval
    sends the query and document tree to the LLM.
    LLM reasons over the tree strucutre and returns the relevant node ids.
    
    Returns a dict with: 'thinking' (reasoning) and node_list(node ids)
    """

    # compress tree to save tokens - only send titles + short summaries
    def compress(nodes):
        compressed = []
        for n in nodes:
            compressed.append({
                "node_id" : n["node_id"],
                "title" : n["title"],
                "summary" : n.get("summary", "")[:200], # first 200 chars
                "page" : n.get("page_index", "?")
            })
            if n.get("nodes"):
                compressed[-1]["nodes"] = compress(n["nodes"])
        return compressed
    
    compressed_tree = compress(tree)

    prompt = f"""You are given a query and a documents tree strcuture (such as tabel of contents).

    Return your response strictly as a valid JSON object.
    Do not include any text markdown wrapping outside the JSON structure.

    Your task: identify which node IDs most likely contain the answer to the query.
    Think step by step about which section are most relevant.
    
    Query: {query}

    Document Tree:
    {json.dumps(compressed_tree, indent = 2)}

    Follow this exact JSON schema format:
    {{
    "thinking": "your step by step reasoning about which sections are relevant",
    "node_list": ["list of node ids you want to retrieve, in order of relevance"]
    }}"""

    response = groq_client.chat.completions.create(
        model = model,
        messages = [{"role": "user", "content": prompt}],
        response_format = {"type": "json_object"}
    )

    return json.loads(response.choices[0].message.content)




In [54]:
query = "Who directed the DDLJ?"

result = llm_tree_search(query, pageindex_tree)
print(result.get("thinking", "No reasoning provided"))
print("Nodes to retrieve:", result.get("node_list", []))


The query asks for the director of the DDLJ. Based on the title of each node, Chapter 5: Cultural Impact and Legacy - Dilwale Dulhania Le Jayenge's Enduring Appeal contains the information about the making and enduring appeal of the film which suggests information about the person behind the film, but Chapter 6: Behind the Scenes Stories - Anecdotes and Unseen Efforts provides more about the directorial debut of the person behind DDLJ's making. Therefore, the relevant node ids are those from chapter 6.
Nodes to retrieve: ['0010']


### Full End-to-End RAG Pipeline

3 steps:

1. Tree Search → LLM picks relevant node_ids
2. Retrieve → Fetch the actual section content from those nodes
3. Generate → LLM writes a grounded answer with page citations

What makes this better than vector RAG:

- Retrieved content has titles + page numbers (traceable)
- LLM can cite exactly which section the answer comes from
- No hallucination from irrelevant chunks

In [55]:
# Helper: Find nodes by ID

def find_nodes_by_ids(tree: list, target_ids: list) -> list:
    """Recursively search the tree for nodes with IDs in target ids."""
    found_nodes = []
    for node in tree:
        if node["node_id"] in target_ids:
            found_nodes.append(node)
        if node.get("nodes"):
            found_nodes.extend(find_nodes_by_ids(node["nodes"], target_ids))
    return found_nodes

In [56]:
# Generate answer for retrieved nodes

def generate_answer(query: str, nodes: list, model: str = "llama-3.1-8b-instant") -> str:
    """
    Takes retireved nodes as a context and generate the grounded answer.
    Instructs the LLM to cite section titles and page numbers.
    """
    if not nodes:
        return "⚠️ No relevant section found in the document."
    
    context_parts = []
    for node in nodes:
        content = node.get("content", "")
        title = node.get("title", "Untitled")
        page = node.get("page_index", "?")
        context_parts.append(f"Section: {title} (p.{page})\n{content}\n")
    context = "\n---\n".join(context_parts)

    prompt = """You are an expert document analyst.
    Answer the question based solely on the provided context.
    For each fact you state, cite the section title and page number from which it was derived.
    Be concise and precise in your answer.
    
    Question: {query}
    
    Context: {context}
    
    Answer:"""

    response = groq_client.chat.completions.create(
        model = model,
        messages = [{"role": "user", "content": prompt.format(query=query, context=context)}]
    )

    return response.choices[0].message.content.strip()

In [68]:
# complete vectorless RAG flow

def vectorless_rag(query: str, tree: list, verbose: bool = True) -> str:
    """
    Executes the full vectorless RAG flow:
    1. LLM tree search to identify relevant nodes.
    2. Retrieve content for those nodes.
    3. Generate final answer with citations.
    """

    if verbose:
        print(f"🔍 Running LLM tree search for query: '{query}'\n")

    search_result = llm_tree_search(query, tree)
    node_ids = search_result.get("node_list", [])
    if verbose:
        print("LLM Tree Search Result:")
        print(search_result.get("thinking", "No reasoning provided")[:200])
        print(f"Node IDs to retrieve: {node_ids} \n")

    nodes = find_nodes_by_ids(tree, node_ids)

    if verbose:
        print(f"section found for retireval: {[n['title'] for n in nodes]}\n")

    answer = generate_answer(query, nodes)

    if verbose:
        print(f"Retrieved {len(nodes)} nodes. Generating answer...\n")
        print("Answer:\n", answer)

    return answer

In [70]:
# run the full pipeline
answer = vectorless_rag(query = "Who directed the DDLJ?",
                        tree = pageindex_tree)

🔍 Running LLM tree search for query: 'Who directed the DDLJ?'

LLM Tree Search Result:
First, I identify the query as 'Who directed the DDLJ?' and look for words or phrases related to the query in the document tree. Then, I notice 'Aditya Chopra' mentioned in node 0005, which is likely 
Node IDs to retrieve: ['0005', '0006', '0010'] 

section found for retireval: ['Chapter 1: The Making of a Modern Classic - Origins and Inspirations', 'Chapter 2: Casting Choices and Iconic Performances - Building Unforgettable Characters', 'Chapter 6 : Behind the Scenes Stories - Anecdotes and Unseen Efforts']

Retrieved 3 nodes. Generating answer...

Answer:
 Aditya Chopra directed the movie Dilwale Dulhania Le Jayenge (DDLJ). 

No specific page number is provided in the given sections for this information.
